# 04 Перенос SmolVLA на ROCm и взвешивание выборки

        Этот раздел на примере SmolVLA показывает: языковую политику нельзя оценивать только по общей успешности. Инструкции для красной и синей чашки нужно оценивать отдельно, а затем сравнить эффект копирования episode и Weighted sampler.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: результаты фиксированных инструкций для красной и синей чашки


In [ ]:
snapshot = json.loads((ASSET_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
rows = []
for item in snapshot["smolvla_forced_instruction_n10"]:
    rows.append((item["label"], item["red_text"], item["blue_text"]))
md_table(["Стратегия", "physical_success для красной чашки", "physical_success для синей чашки"], rows)
show_asset("smolvla_red_blue_success.png", width=1100)


У baseline хорошо с красной чашкой и плохо с синей: модель не то чтобы совсем не умеет захватывать, а распределение, связанное с условием задачи и цветом, ещё не выучено устойчиво.


## Checkpoint 2: разница между копированием episode и Weighted sampler


In [ ]:
rows = [
    ("Копирование episode", "напрямую меняет распределение episode в наборе данных", "просто реализовать, но модель легко смещается к одному цвету"),
    ("Weighted sampler", "не меняет исходный parquet, только вероятности выборки", "проще откатить и лучше подходит для балансировки малых данных"),
    ("Оценка промежуточных checkpoint", "не смотрите только на final step", "в этом примере step500 сбалансированнее step1000"),
]
md_table(["Метод", "Как делается", "На что обратить внимание"], rows)


## Checkpoint 3: пересоздание графиков


In [ ]:
cmd = [
    sys.executable,
    str(TOPIC_ROOT / "code" / "generate_tutorial_assets.py"),
    "--source-root",
    str(OUTPUT_ROOT),
]
print("Если в OUTPUT_ROOT есть нужные этой главе TSV/JSONL/MP4, можно запустить:")
print(" ".join(cmd))
# subprocess.run(cmd, check=True)


Ячейка выше по умолчанию не выполняется, чтобы не создавать сводные графики метрик, пока выходной каталог не готов. Убедившись, что `$OUTPUT_ROOT` указан верно, раскомментируйте последнюю строку.


## Checkpoint 4: сравнение провала и успеха после исправления для синей чашки


In [ ]:
show_asset("smolvla_blue_failure_sequence.jpg", width=1100)
show_asset("smolvla_blue_success_sequence.jpg", width=1100)
